In [ ]:
# Radiacode full pipeline (Jupyter version)
# • Extract count-rate rows from a .zip of .rctrk tracks
# • Prune points whose implied speed > 6 km/h
# • Save raw and cleaned CSVs (optional)

import zipfile, json, csv, io, os, math
from datetime import datetime
import pandas as pd

# ---------------- Configuration ----------------
SPEED_LIMIT_KMH = 6.0               # pruning threshold
RAW_CSV   = "radiacode_raw.csv"
CLEAN_CSV = "radiacode_clean_speed.csv"

COUNT_KEYS = ("countRate", "count_rate", "CountRate")
DOSE_KEYS  = ("doseRate",  "dose_rate",  "DoseRate")  # not used here, but kept for completeness

# ---------------- Helpers ----------------
def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance (km) on WGS-84 sphere."""
    R = 6371.0
    φ1, φ2 = math.radians(lat1), math.radians(lat2)
    dφ     = math.radians(lat2 - lat1)
    dλ     = math.radians(lon2 - lon1)
    a = math.sin(dφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(dλ/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def get_first(d, keys):
    """Return the first non-None, non-empty value for keys in dict-like d."""
    for k in keys:
        if k in d and d[k] not in ("", None):
            return d[k]
    return None

def derive_label(fname):
    return os.path.splitext(os.path.basename(fname))[0].strip()

# ---------------- Stage 1 – read ZIP and build raw CPS table ----------------
def build_raw_df(zip_path: str) -> pd.DataFrame:
    rows = []

    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if not name.lower().endswith(".rctrk"):
                continue
            if name.startswith((".__", "__MACOSX/", "._")):
                continue  # macOS resource forks etc.

            data = zf.read(name)

            # JSON-style track
            if data.lstrip().startswith(b"{"):
                obj = json.loads(data.decode("utf-8", "replace"))
                markers = obj.get("markers") or obj.get("data") or obj.get("rows") or []
                for m in markers:
                    cr = get_first(m, COUNT_KEYS)
                    if cr is None:
                        continue  # skip dose-only rows

                    # timestamp handling: epoch seconds OR ISO-like string
                    ts = m.get("time") or m.get("timestamp")
                    dt = None
                    if isinstance(ts, (int, float)):
                        try:
                            dt = datetime.utcfromtimestamp(ts)
                        except Exception:
                            dt = None
                    elif isinstance(ts, str):
                        dt = pd.to_datetime(ts, utc=True, errors="coerce")
                        if pd.notna(dt):
                            dt = dt.tz_convert("UTC").tz_localize(None) if getattr(dt, "tzinfo", None) else dt.to_pydatetime()
                        else:
                            dt = None

                    date_s = dt.strftime("%Y-%m-%d") if dt else ""
                    time_s = dt.strftime("%H:%M:%S") if dt else ""

                    # coords
                    lat = m.get("lat") or m.get("latitude")
                    lon = m.get("lon") or m.get("longitude")
                    try:
                        lat = float(lat) if lat is not None else None
                        lon = float(lon) if lon is not None else None
                        crf = float(cr)
                    except Exception:
                        continue

                    rows.append(
                        dict(
                            source_file = name,
                            park        = derive_label(name),
                            lat         = lat,
                            lon         = lon,
                            activity    = crf,
                            date        = date_s,
                            time        = time_s,
                        )
                    )

            # Delimited text style (.csv/.tsv)
            else:
                text = data.decode("utf-8", "replace")
                if not text.strip():
                    continue
                first_line = text.splitlines()[0] if text.splitlines() else ""
                delim = "\t" if "\t" in first_line else ","
                rdr = csv.DictReader(io.StringIO(text), delimiter=delim)

                for r in rdr:
                    cr = get_first(r, COUNT_KEYS)
                    if cr is None:
                        continue
                    try:
                        cr_val = float(cr)
                    except ValueError:
                        continue

                    lat = r.get("lat") or r.get("latitude")
                    lon = r.get("lon") or r.get("longitude")
                    try:
                        lat = float(lat) if lat is not None else None
                        lon = float(lon) if lon is not None else None
                    except Exception:
                        continue

                    rows.append(
                        dict(
                            source_file = name,
                            park        = derive_label(name),
                            lat         = lat,
                            lon         = lon,
                            activity    = cr_val,
                            date        = (r.get("date") or "").strip(),
                            time        = (r.get("time") or "").strip(),
                        )
                    )

    df = pd.DataFrame(rows)
    return df

# ---------------- Stage 2 – prune points moving > SPEED_LIMIT_KMH ----------------
def prune_by_speed(df: pd.DataFrame, speed_limit_kmh: float = SPEED_LIMIT_KMH) -> pd.DataFrame:
    ts = pd.to_datetime((df["date"].fillna("").astype(str) + " " + df["time"].fillna("").astype(str)).str.strip(),
                        errors="coerce", utc=False)
    df = df.copy()
    df["timestamp"] = ts

    keep_rows = []

    for _, grp in df.groupby("source_file", sort=False):
        g = grp.sort_values("timestamp").reset_index(drop=True)

        if len(g) == 0:
            continue
        if len(g) == 1:
            keep_rows.append(g.iloc[[0]])
            continue

        keep_mask = [True]  # keep the first point
        for i in range(1, len(g)):
            lat1, lon1, t1 = g.loc[i-1, ["lat","lon","timestamp"]]
            lat2, lon2, t2 = g.loc[i,   ["lat","lon","timestamp"]]

            # require valid coords and strictly increasing time
            if (pd.isna(t1) or pd.isna(t2) or t1 == t2 or
                pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2)):
                keep_mask.append(False)
                continue

            dt_h  = (t2 - t1).total_seconds() / 3600.0
            if dt_h <= 0:
                keep_mask.append(False)
                continue

            dist  = haversine(lat1, lon1, lat2, lon2)
            speed = dist / dt_h
            keep_mask.append(speed <= speed_limit_kmh)

        keep_rows.append(g[keep_mask])

    clean = pd.concat(keep_rows, ignore_index=True) if keep_rows else pd.DataFrame(columns=df.columns)
    return clean.drop(columns="timestamp", errors="ignore")

# ---------------- Convenience runner for notebooks ----------------
def run_pipeline(zip_path: str, save_csv: bool = True,
                 raw_csv: str = RAW_CSV, clean_csv: str = CLEAN_CSV):
    raw_df = build_raw_df(zip_path)
    clean_df = prune_by_speed(raw_df)

    if save_csv:
        raw_df.to_csv(raw_csv, index=False)
        clean_df.to_csv(clean_csv, index=False)

    print(f"Raw count-rate rows : {len(raw_df):>6}")
    print(f"After speed filter  : {len(clean_df):>6}")
    if save_csv:
        print("CSV files saved:\n  ", raw_csv, "\n  ", clean_csv)

    return raw_df, clean_df

In [ ]:
# Set your .zip path and run
ZIP_PATH = "OneDrive_1_7-8-2025 2.zip"  # <- change me

raw_df, clean_df = run_pipeline(ZIP_PATH, save_csv=True)

# Quick peek
display(raw_df.head())
display(clean_df.head())